# Wordle 训练使用的 TorchTitan-NPU 特性

Wordle 训练会用到 TorchTitan-NPU 的两卡 FSDP2、CPU offload、Qwen3 packed 变长序列计算和分片状态接口。本节沿着一次训练计算展开这些配置，重点看 TND 变长注意力怎样进入 Actor 与 Ref。


## 内存与计算策略

| 配置或策略 | 本案例值 | 主要作用 | 需要注意的取舍 |
| --- | --- | --- | --- |
| `param_offload` | Actor/Ref：True | 空闲阶段将参数迁移到 CPU | 增加 host-device 搬移 |
| `optimizer_offload` | Actor：True | 空闲阶段迁移 Actor 优化器状态 | 更新前需要恢复状态；Ref 不创建优化器 |
| `reshard_after_forward` | Actor：always | forward 后及时恢复 Actor 分片状态 | 反向前可能需要再次通信 |
| TorchTitan 训练精度 | BF16 参数、FP32 reduction | 降低参数计算开销并保持归约精度 | 需要关注数值稳定性 |
| `enable_gradient_checkpointing` | True | 反向时重算部分激活 | 用额外计算换取激活显存空间 |

Ref 为 forward-only 模型，使用参数 offload，但没有优化器状态；Actor 的 `reshard_after_forward=always` 也不应外推为 Ref 配置。TorchTitan 的训练精度由 `TrainingConfig` 默认值确定，不依赖额外的 Hydra 布尔开关。

这些设置主要解决显存占用和计算路径问题，也为增大 batch、序列或模型留出潜在空间。吞吐是否变化，还取决于模型、序列分布以及 CPU/NPU 搬移和通信开销。

训练状态可能同时由模型、优化器和 checkpoint manager 持有。执行 CPU offload 时，需要确保这些对象不再保留训练阶段的 NPU tensor 引用，才能为后续 vLLM rollout 释放足够显存。


<img src="./images/npu_converter_flow.png" alt="TorchTitan-NPU Qwen3 模型适配" width="90%">

## Qwen3 的 NPU 模型适配与 TND 布局

TorchTitan-NPU converter 在模型构建阶段注册 `npu_rms_norm` 和 `npu_rope`，将对应 Qwen3 模块替换为适合 Ascend NPU 的实现。转换发生在明确的模块边界，不改变模型层数、隐藏维度、词表或 Wordle 任务输入输出。

packed batch 的注意力路径通过 Qwen3 model spec 单独启用 `NPUVarlenAttention`。启动配置使用 `attn_type=varlen`，并将单条样本的 Actor 与 rollout 最大长度统一为 5120，即 1024 token prompt 与 4096 token response 的总预算。

进入注意力算子前，Q、K、V 从 BSND 视图整理为 TND 布局：

| 维度 | 含义 | 在 packed batch 中的作用 |
| --- | --- | --- |
| T | 当前 packed micro-batch 的总 token 数 | 多条样本移除 padding 后首尾拼接 |
| N | 注意力头数 | 分别承载 query heads 与 KV heads |
| D | 每个注意力头的 head dimension | 保持模型结构定义 |

TND 只描述张量布局，不能单独表达每条样本从哪里开始和结束。适配层还会构造 `VarlenMetadata`，把累计序列长度传给 CANN FA v3；`sparse_mode=7` 按这些边界执行逐样本因果注意力，避免相邻 packed 样本互相看到 token。


## TND 为什么适合强化学习训练

Wordle AgentLoop 可能在不同轮次结束，每条 response 的有效长度并不相同。如果把同一 micro-batch 补齐到最长序列，较短样本会引入大量无效 token。TND 变长路径按以下过程处理这些样本：

1. 移除样本 padding，形成 packed token 流和原始 offsets；
2. 使用 offsets 构造 `VarlenMetadata`，记录每条样本的累计序列长度；
3. 将 Q、K、V 整理为 TND 布局；
4. CANN FA v3 根据边界元数据执行逐样本因果注意力；
5. 按原始 offsets 恢复 jagged tensor，继续计算 log-prob、entropy 和 GRPO loss。

这条路径减少的是 padding 带来的无效计算，不改变单条样本的上下文上限。`ppo_max_token_len_per_gpu=5120` 约束每张 NPU 的 packed token 容量；单条样本仍受 1024 token prompt 与 4096 token response 组成的 5120 token 上限约束。若在更长序列场景中进一步组合 CP，还需要额外处理 token 补齐、切分、all-to-all 和结果还原，本案例不启用该路径。


## 状态接口与权重同步

TorchTitan FSDP2 下，Actor 权重分布在两个 rank。Verl 不能假定每个 rank 都持有完整的普通 tensor，而是通过训练 engine 的 `get_per_tensor_param()` 读取分片权重。`StateDictAdapter` 负责将参数名映射到 vLLM 所需的 Hugging Face 格式，DTensor 分片则在传输前物化为完整 tensor。

完成优化器更新后，unified worker 调用 rollout 的 `update_weights()` 把最新 Actor 权重同步给 vLLM。这个过程既要保持参数名称和形状一致，也要正确处理 DTensor 分片；否则训练端虽然更新成功，生成端仍可能继续使用旧策略。

第 7 章将沿用 DP shard 2、CP 1 的 FSDP2 + TND 配置完成三步训练。`torch.compile`、TP、PP 和 EP 均保持关闭。


## 课后练习

### 判断题

1. （判断题）TND 中的 T 表示 packed micro-batch 的总 token 数。

2. （判断题）TND 布局本身已经包含每条 packed 样本的边界，因此不再需要 VarlenMetadata。

### 单选题

3. （单选题）`sparse_mode=7` 与累计序列长度在本案例中共同保证什么？

   A. 每条 packed 样本独立执行因果注意力

   B. 每个 rank 使用不同 tokenizer

   C. 自动增加 CP degree

   D. 关闭参数卸载

### 多选题

4. （多选题）本案例的 TND 变长计算路径包含哪些步骤？

   A. 移除 padding 并拼接有效 token

   B. 构造累计序列长度元数据

   C. 执行逐样本因果注意力

   D. 按 offsets 恢复 jagged tensor

5. （多选题）Actor 状态接口在 RL 流水线中承担哪些作用？

   A. 提供更新后的训练权重

   B. 支持权重同步到 vLLM

   C. 重新定义 Wordle 奖励

   D. 适配分片状态的读取


> 完成练习后，运行下方单元格查看参考答案和解析。


In [ ]:
from pathlib import Path
import subprocess

course_root = Path(subprocess.check_output(['git', 'rev-parse', '--show-toplevel'], text=True).strip())
answer_path = course_root / 'tutorials/rl_training_pipeline/06_torchtitan_npu_features/answer/06.03_answer.txt'
assert answer_path.is_file(), f'未找到答案文件: {answer_path}'
print(answer_path.read_text(encoding='utf-8'))
